# Module 6, Lab 1: LangChain Basics
## Simple LLM App for Routine Corporate Operations

**Objective:** Learn the LangChain basics by building small AI helpers for everyday office and operations work: shift handovers, helpdesk tickets, meeting notes and status emails.

**Prerequisites:**
- Python 3.10 or higher
- An OpenAI API key in the `.env` file in this folder
- Sample data only. Do not paste real employee, customer or financial records.

**Duration:** 2-3 hours

## Section 1: Setup and Installation

In [1]:
# Install once in a terminal, not in this notebook:
#   pip install -r lab1_m6_requirements.txt jupyter
import os
import json
from dotenv import load_dotenv

# Load settings from .env_lab1_m6 (if you made one) and from .env in this folder
load_dotenv(".env_lab1_m6")
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

openai_key = os.getenv("OPENAI_API_KEY")

print("LangChain Lab 1 setup complete")
print("OpenAI API key found:", bool(openai_key))

LangChain Lab 1 setup complete
OpenAI API key found: True


## Section 2: How the Pieces Fit Together

In [2]:
building_blocks = {
    "LLM (ChatOpenAI)": "Sends a request to the AI model and gets a reply",
    "PromptTemplate": "A reusable instruction with blanks, such as {ticket_text}",
    "ChatPromptTemplate": "A prompt split into system and human messages",
    "Output parser": "Turns the model reply into plain text or JSON",
    "Chain: prompt | llm | parser": "Runs the steps in order, passing each output to the next step",
}

print("LANGCHAIN BUILDING BLOCKS")
print("=" * 70)
for name, description in building_blocks.items():
    print(f"{name:<30} {description}")
print("=" * 70)
print("Every chain in this lab uses the pattern: prompt | llm | parser")

LANGCHAIN BUILDING BLOCKS
LLM (ChatOpenAI)               Sends a request to the AI model and gets a reply
PromptTemplate                 A reusable instruction with blanks, such as {ticket_text}
ChatPromptTemplate             A prompt split into system and human messages
Output parser                  Turns the model reply into plain text or JSON
Chain: prompt | llm | parser   Runs the steps in order, passing each output to the next step
Every chain in this lab uses the pattern: prompt | llm | parser


## Section 3: Working with the LLM

The model name, temperature and token limit are read from the `.env` file. If a setting is missing, the default in the code is used.

In [3]:
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
LLM_TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", "0.2"))
MAX_TOKENS = int(os.getenv("MAX_TOKENS", "1000"))

llm = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=LLM_TEMPERATURE,
    max_tokens=MAX_TOKENS,
    api_key=openai_key,
)

print(f"Model: {OPENAI_MODEL} | temperature: {LLM_TEMPERATURE} | max tokens: {MAX_TOKENS}")

Model: gpt-4o-mini | temperature: 0.7 | max tokens: 1000


### 3.1: Basic LLM Call

In [4]:
question = "In one sentence, what is a shift handover in an operations team?"
reply = llm.invoke(question)

print("Q:", question)
print("A:", reply.content)

Q: In one sentence, what is a shift handover in an operations team?
A: A shift handover in an operations team is the process of transferring responsibility and important information about ongoing tasks, issues, and status updates from one shift to the next to ensure continuity and operational efficiency.


## Section 4: Prompt Templates

In [5]:
simple_prompt = PromptTemplate(
    input_variables=["note"],
    template="Rewrite this note as one clear sentence for a manager:\n\n{note}\n",
)

print(simple_prompt.format(note="Printer on floor 2 jammed again, ticket FAC-882 open."))

Rewrite this note as one clear sentence for a manager:

Printer on floor 2 jammed again, ticket FAC-882 open.



### 4.1: Template with Several Steps (Shift Handover)

In [6]:
handover_prompt = PromptTemplate(
    input_variables=["shift_notes"],
    template="""You are an operations assistant.
Summarize these shift notes for the next shift.
Give: 1) Open issues  2) Actions for the next shift  3) Items completed.
Keep it under 120 words.

Shift notes:
{shift_notes}
""",
)

print(handover_prompt.format(shift_notes="Printer on floor 2 jammed."))

You are an operations assistant.
Summarize these shift notes for the next shift.
Give: 1) Open issues  2) Actions for the next shift  3) Items completed.
Keep it under 120 words.

Shift notes:
Printer on floor 2 jammed.



### 4.2: Chat Prompt Template (Roles)

In [7]:
ticket_chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an office helpdesk triage assistant. Answer briefly."),
    ("human", "Ticket: {ticket_text}\nWhich team should handle it: IT, Facilities, HR, or Finance? Reply with the team name and one short reason."),
])

for message in ticket_chat_prompt.format_messages(ticket_text="Air conditioning on floor 3 is broken."):
    print(message.type.upper(), "->", message.content)

SYSTEM -> You are an office helpdesk triage assistant. Answer briefly.
HUMAN -> Ticket: Air conditioning on floor 3 is broken.
Which team should handle it: IT, Facilities, HR, or Finance? Reply with the team name and one short reason.


## Section 5: Chains (prompt | llm | parser)

A chain is a prompt, then the model, then a parser that returns plain text. This section runs the shift handover chain on the sample shift notes.

In [8]:
handover_chain = handover_prompt | llm | StrOutputParser()

with open("data/sample_shift_notes.txt", encoding="utf-8") as f:
    shift_notes = f.read()

handover_summary = handover_chain.invoke({"shift_notes": shift_notes})
print(handover_summary)

**Open Issues:**  
- Printer jam on floor 2 (ticket FAC-882 open).  
- Vendor portal login is slow (monitoring ongoing).  
- Badge reader at loading dock offline (waiting for security vendor callback).  

**Actions for Next Shift:**  
- Follow up on badge reader issue.  
- Check status of vendor portal ticket.  
- Confirm patch notice was sent to all staff.  

**Items Completed:**  
- Backup job for finance server re-ran and succeeded.  
- Weekly patch window confirmed for Saturday at 22:00.  


## Section 6: Multi-Step Chain (Pass One Output to the Next Step)

In [9]:
action_prompt = PromptTemplate(
    input_variables=["summary"],
    template="""From this shift handover summary, list the next-shift tasks.
Use a numbered list. Add the owner role (IT, Facilities, Security, or Finance) for each task.

Summary:
{summary}
""",
)

action_chain = action_prompt | llm | StrOutputParser()
next_shift_tasks = action_chain.invoke({"summary": handover_summary})
print(next_shift_tasks)

Next-shift tasks:

1. Follow up on badge reader issue - Owner: Security  
2. Check status of vendor portal ticket - Owner: IT  
3. Confirm patch notice was sent to all staff - Owner: IT  


## Section 7: Output Parsing (JSON with a Fallback)

Models do not always return clean JSON. The helper below removes code fences and returns `None` if the reply still cannot be read, so the code can handle the failure.

In [10]:
def parse_json_reply(raw_text):
    """Remove code fences, then parse JSON. Returns None if the text is not valid JSON."""
    cleaned = raw_text.strip().replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return None


triage_prompt = PromptTemplate(
    input_variables=["ticket_text"],
    template="""Classify this office or IT ticket.
Return ONLY a JSON object with these keys:
"team" (IT, Facilities, HR, or Finance), "priority" (Low, Medium, or High), "reason" (one short sentence).

Ticket: {ticket_text}
""",
)

triage_chain = triage_prompt | llm | StrOutputParser()

raw = triage_chain.invoke({"ticket_text": "Printer on floor 4 is offline and the team cannot print client invoices."})
result = parse_json_reply(raw)

if isinstance(result, dict):
    print(result.get("team"), "|", result.get("priority"), "|", result.get("reason"))
else:
    print("Not valid JSON. Raw output:")
    print(raw)

IT | High | Printer on floor 4 is offline affecting client invoice printing.


## Section 8: Hands-On Exercise: Meeting Status Email

The starter prompt below writes a status email with a subject line. Your task:
1. Change the instructions so the email also has **3 bullet points** and **one line listing owners and due dates**.
2. Run the cell again and compare the two results.
3. Try a lower and a higher `LLM_TEMPERATURE` in `.env` and see how the wording changes.

In [11]:
meeting_notes = open("data/sample_meeting_notes.txt", encoding="utf-8").read()

# TODO: change these instructions to add the 3 bullet points and the owners-and-dates line.
status_prompt = PromptTemplate(
    input_variables=["meeting_notes"],
    template="""Write a short status email from these meeting notes.
Start with a subject line.

Meeting notes:
{meeting_notes}
""",
)

status_chain = status_prompt | llm | StrOutputParser()
print(status_chain.invoke({"meeting_notes": meeting_notes}))

Subject: Weekly Operations Meeting Update

Dear Team,

Here are the key takeaways from our Weekly Operations Meeting:

1. **Office Move**: The transition to floor 4 is on track, with desks ready by the 28th. Facilities will confirm the status of network points.
   
2. **Invoice Backlog**: There are currently 14 invoices pending approval. Finance is committed to clearing these by Wednesday.

3. **Laptop Order**: The new laptop order is expected to arrive on Thursday. IT will handle imaging and labeling before Friday's onboarding.

4. **Vendor Contract Renewal**: The renewal is due next month. Procurement will provide a comparison by the 20th.

Please let me know if you have any questions or need further details.

Best,  
[Your Name]  
[Your Position]


## Section 9: Best Practices and Tips

In [12]:
best_practices = {
    "Prompt Design": [
        "Be specific: say who the audience is and what format you want",
        "Ask for a fixed output format (JSON or a numbered list) so you can parse it",
        "Give the model only the information it needs",
    ],
    "Chain Design": [
        "Break a job into small steps, one chain per step",
        "Test each chain on one example before running a batch",
        "Keep the output of each step, so you can check it",
    ],
    "Error Handling": [
        "Always keep a fallback for replies that do not parse",
        "Check the results before you act on them",
    ],
    "Cost and Privacy": [
        "Start with the default model (gpt-4o-mini) and move up only if quality is not good enough",
        "Use sample data only. Do not paste real employee, customer or financial records",
    ],
}

print("BEST PRACTICES")
print("=" * 70)
for category, tips in best_practices.items():
    print(f"\n{category.upper()}:")
    for tip in tips:
        print(f"  - {tip}")
print("\n" + "=" * 70)

BEST PRACTICES

PROMPT DESIGN:
  - Be specific: say who the audience is and what format you want
  - Ask for a fixed output format (JSON or a numbered list) so you can parse it
  - Give the model only the information it needs

CHAIN DESIGN:
  - Break a job into small steps, one chain per step
  - Test each chain on one example before running a batch
  - Keep the output of each step, so you can check it

ERROR HANDLING:
  - Always keep a fallback for replies that do not parse
  - Check the results before you act on them

COST AND PRIVACY:
  - Start with the default model (gpt-4o-mini) and move up only if quality is not good enough
  - Use sample data only. Do not paste real employee, customer or financial records



## Section 10: Common Routine-Operations Patterns

In [13]:
common_patterns = {
    "Shift handover summary": {
        "input": "Shift log notes",
        "output": "Open issues and next-shift actions",
    },
    "Ticket triage": {
        "input": "One helpdesk or office ticket",
        "output": "Team, priority and reason (JSON)",
    },
    "Meeting action items": {
        "input": "Meeting notes",
        "output": "Owners, tasks and due dates",
    },
    "Weekly status report": {
        "input": "Task updates",
        "output": "Short status email",
    },
}

for pattern, details in common_patterns.items():
    print(f"{pattern}")
    print(f"   Input:  {details['input']}")
    print(f"   Output: {details['output']}")

Shift handover summary
   Input:  Shift log notes
   Output: Open issues and next-shift actions
Ticket triage
   Input:  One helpdesk or office ticket
   Output: Team, priority and reason (JSON)
Meeting action items
   Input:  Meeting notes
   Output: Owners, tasks and due dates
Weekly status report
   Input:  Task updates
   Output: Short status email


## Section 11: Complete Example: Batch Ticket Triage

This section loads the sample tickets, triages each one with the chain from Section 7, and saves the results to `output/ticket_triage.json`. It uses plain functions only.

In [14]:
def load_tickets(filepath):
    """Read the list of tickets from a JSON file."""
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)["tickets"]


def triage_one_ticket(chain, ticket):
    """Run the triage chain on one ticket and return a row for the results file."""
    raw = chain.invoke({"ticket_text": ticket["text"]})
    parsed = parse_json_reply(raw)
    if not isinstance(parsed, dict):
        return {"id": ticket["id"], "team": "UNPARSED", "priority": "UNPARSED", "reason": raw.strip()}
    return {
        "id": ticket["id"],
        "team": parsed.get("team"),
        "priority": parsed.get("priority"),
        "reason": parsed.get("reason"),
    }


def save_results(results, filepath):
    """Save the results as JSON, creating the folder if needed."""
    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)


tickets = load_tickets("data/sample_tickets.json")
triage_results = []

for ticket in tickets:
    row = triage_one_ticket(triage_chain, ticket)
    triage_results.append(row)
    print(row["id"], row["team"], row["priority"])

save_results(triage_results, "output/ticket_triage.json")
print(f"Saved {len(triage_results)} results to output/ticket_triage.json")

TKT-101 IT High


TKT-102 Facilities High


TKT-103 HR Medium


TKT-104 Finance Medium


TKT-105 IT High


TKT-106 Facilities Medium


TKT-107 IT High


TKT-108 Finance High
Saved 8 results to output/ticket_triage.json


## Section 12: Summary and Next Steps

You learned how to:
- start a `ChatOpenAI` model with settings from `.env`
- write `PromptTemplate` and `ChatPromptTemplate` prompts for office work
- build chains with `prompt | llm | StrOutputParser()`, as one step and as several steps
- parse structured output with a fallback
- process a batch of routine tickets and save the results

**Next labs:**
- Lab 2: sequential chains for routine operations requests
- Lab 3: conversational chains for routine operations issues

In [15]:
summary = {
    "Concepts covered": [
        "LangChain building blocks and the prompt | llm | parser pattern",
        "Settings from .env (model, temperature, token limit)",
        "Prompt templates for office and operations tasks",
        "Single-step and multi-step chains",
        "JSON output with a fallback",
        "Batch processing and saving results",
    ],
    "Try next": [
        "Write a prompt for a weekly status report",
        "Add a new ticket category to the triage prompt",
        "Compare results at two temperatures",
    ],
}

for section, items in summary.items():
    print(section)
    for item in items:
        print(f"  - {item}")
print("\nLab 1 complete.")

Concepts covered
  - LangChain building blocks and the prompt | llm | parser pattern
  - Settings from .env (model, temperature, token limit)
  - Prompt templates for office and operations tasks
  - Single-step and multi-step chains
  - JSON output with a fallback
  - Batch processing and saving results
Try next
  - Write a prompt for a weekly status report
  - Add a new ticket category to the triage prompt
  - Compare results at two temperatures

Lab 1 complete.
